In [14]:
import os
import h5py
import numpy as np

In [15]:
def collect_h5_file_paths(root_dir):
    file_paths = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith('h5'):
                file_paths.append(os.path.join(dirpath, filename))

    return file_paths

In [16]:
ROOT_DIR = './data'
file_paths = collect_h5_file_paths(ROOT_DIR)

In [17]:
len(file_paths)

10000

In [18]:
def extract_features_from_h5(h5_file_path):
    """
    從 Million Song Dataset 的 .h5 檔案中提取歌曲 ID 和 24 維音色特徵向量 (Mean/Std)。
    
    參數:
        h5_file_path (str): HDF5 檔案的完整路徑。

    返回:
        tuple (str, numpy.ndarray) 或 (None, None): 
        包含 (song_id, feature_vector) 或 (None, None)。
    """
    try:
        with h5py.File(h5_file_path, 'r') as f:
            # 檢查關鍵群組是否存在
            if 'metadata' not in f or 'analysis' not in f:
                raise KeyError("Required groups 'metadata' or 'analysis' not found.")
            
            # 1. 提取歌曲 ID (從複合數據集 'songs' 中)
            # 假設 'songs' 是唯一的複合數據集，且 'song_id' 在其中
            songs_metadata = f['metadata']['songs']
            # 提取第一個記錄中的 'song_id' 欄位，並將其解碼為 Python 字串
            song_id = songs_metadata['song_id'][0].decode('utf-8')

            # 2. 提取 segments_timbre 矩陣
            # 這是位於 'analysis' 群組下的數據集
            timbre = f['analysis']['segments_timbre'][:]
            
            # 3. 彙總計算 (Mean and Std)
            # Timbre 應有 12 維。計算每維的平均值 (12 維) 和標準差 (12 維)。
            timbre_mean = np.mean(timbre, axis=0)
            timbre_std = np.std(timbre, axis=0)
            
            if len(timbre_mean) != 12:
                 raise ValueError(f"Timbre dimension mismatch: expected 12, got {len(timbre_mean)}")

            # 4. 拼接成單一 24 維特徵向量
            feature_vector = np.concatenate([timbre_mean, timbre_std])
            
            return song_id, feature_vector

    except KeyError as ke:
        # 捕獲因路徑錯誤或數據集/欄位缺失導致的錯誤
        print(f"Skipping error processing {h5_file_path}: Key error: {ke}")
        return None, None
    except IndexError as ie:
        # 捕獲因嘗試對空數據集進行索引 [0] 導致的錯誤
        print(f"Skipping error processing {h5_file_path}: Index error: {ie}")
        return None, None
    except Exception as e:
        # 捕獲其他所有異常
        print(f"Skipping error processing {h5_file_path}: General error: {e}")
        return None, None

In [19]:
all_song_data = {}

# 遍歷所有檔案並提取數據
print("Starting to scan files and extract features...")
for path in file_paths:
    song_id, features = extract_features_from_h5(path)
    
    if song_id:
        all_song_data[song_id] = features
print(f"Finished scanning. Total {len(all_song_data)} songs processed.")

Starting to scan files and extract features...
Finished scanning. Total 10000 songs processed.


In [20]:
all_song_data

{'SOMZWCG12A8C13C480': array([ 41.51277755,  15.66293615,  -5.78968177,  -0.78984449,
        -43.23490319,  15.17094748,  15.13903811,   1.50500103,
          6.58957055,  12.2985551 , -13.56328012,   5.44247992,
          4.94454249,  61.51997339,  58.72095053,  54.4799564 ,
         32.52931687,  37.56815901,  27.00053439,  35.91573311,
         27.00545659,  24.12389234,  24.51061878,  17.7266023 ]),
 'SOCIWDW12A8C13D406': array([43.07103636, -4.03539091, 23.57229273, 12.92357636, -2.54503636,
         5.05239455,  9.23815091, -4.34597455,  5.22410364,  2.93566364,
        -2.75263818,  1.72939636,  5.84745467, 33.20336162, 28.86020588,
        39.90195041, 25.29130459, 29.12232911, 20.09004869, 19.60809283,
        17.55375405, 16.7973382 , 16.09334278, 12.0891049 ]),
 'SOXVLOJ12AB0189215': array([ 45.13081495, -76.82332562,  50.78732918,  13.50989324,
         -6.54925801,  -5.51424199,   1.88517794,  -8.96372954,
          3.75986299,  -5.11795374,   8.4420694 ,   4.47717082,
  

## 特徵矩陣

In [21]:
# 承接上一個步驟的結果：假設您已經運行了提取函數並有了 all_song_data 字典
# 這裡我們需要重新建構 feature_matrix 和 song_ids 列表，確保它們順序一致

# 1. 建立特徵矩陣
song_ids = list(all_song_data.keys())
feature_list = [data for data in all_song_data.values()]
feature_matrix = np.stack(feature_list)

print(f"Feature Matrix Shape: {feature_matrix.shape}")

# 2. 計算餘弦相似度矩陣
def cosine_similarity_matrix(matrix):
    """
    使用 NumPy 計算特徵矩陣中所有向量對的餘弦相似度矩陣。
    """
    # L2 歸一化
    norm = np.linalg.norm(matrix, axis=1, keepdims=True)
    # 避免除以零，將範數為 0 的元素設為一個極小的數
    norm[norm == 0] = 1e-12 
    normalized_matrix = matrix / norm
    
    # 矩陣乘法 (A @ A.T) 即為歸一化後的餘弦相似度
    similarity_matrix = normalized_matrix @ normalized_matrix.T
    
    return similarity_matrix

sim_matrix = cosine_similarity_matrix(feature_matrix)
print(f"Similarity Matrix Shape: {sim_matrix.shape}")

# 創建一個映射，將 song_id 快速映射到矩陣索引
id_to_index = {song_id: index for index, song_id in enumerate(song_ids)}

Feature Matrix Shape: (10000, 24)
Similarity Matrix Shape: (10000, 10000)


In [22]:
def get_content_based_recommendations(target_song_id, sim_matrix, song_ids, id_to_index, top_n=5):
    """
    根據 Content-Based 相似度矩陣，為一首歌曲找到 Top N 個相似的歌曲。
    
    參數:
        target_song_id (str): 用來推薦的歌曲 ID (Seed Song)。
        sim_matrix (np.ndarray): 歌曲對歌曲的相似度矩陣。
        song_ids (list): 所有歌曲 ID 的列表 (順序與矩陣行/列對應)。
        id_to_index (dict): 歌曲 ID 到矩陣索引的映射。
        top_n (int): 要推薦的歌曲數量。

    返回:
        list of tuple: 包含 (相似歌曲 ID, 相似度分數) 的列表。
    """
    if target_song_id not in id_to_index:
        return f"Error: Song ID {target_song_id} not found in dataset."

    # 1. 取得目標歌曲在矩陣中的索引
    target_index = id_to_index[target_song_id]
    
    # 2. 從相似度矩陣中獲取該歌曲的相似度向量 (NumPy 操作)
    similarity_scores = sim_matrix[target_index]
    
    # 3. 將分數與歌曲索引配對
    # scores_with_indices 是一個列表，包含 (相似度分數, 歌曲索引)
    scores_with_indices = list(enumerate(similarity_scores))
    
    # 4. 排序：根據相似度分數由高到低排序 (使用 NumPy 的 argsort 更高效)
    # 由於我們只關心該行，可以使用 NumPy 的內建排序
    
    # 獲取排序後的索引 (從高到低)
    # np.argsort 默認從小到大，使用 [::-1] 反轉
    sorted_indices = np.argsort(similarity_scores)[::-1]
    
    # 5. 篩選：跳過自己 (相似度為 1.0 的第一首) 和只取 Top N
    
    # 推薦列表：從第二個開始取 Top N
    recommendations = []
    for i in sorted_indices:
        if i != target_index: # 跳過自己
            recommended_song_id = song_ids[i]
            score = similarity_scores[i]
            recommendations.append((recommended_song_id, score))
            
            if len(recommendations) >= top_n:
                break
                
    return recommendations

In [23]:
filename = './data/MillionSongSubset/A/A/A/TRAAAAW128F429D538.h5'
song_id_test, feature_test = extract_features_from_h5(filename)
song_id_test

'SOMZWCG12A8C13C480'

In [24]:
recommendations = get_content_based_recommendations(
    target_song_id=song_id_test,
    sim_matrix=sim_matrix,
    song_ids=song_ids,
    id_to_index=id_to_index,
    top_n=7
)

# 打印結果
print(f"Content-Based Recommendations for {song_id_test} (Top 7):")
for rank, (rec_id, score) in enumerate(recommendations, 1):
    print(f"Rank {rank}: Song ID {rec_id} (Similarity: {score:.4f})")

Content-Based Recommendations for SOMZWCG12A8C13C480 (Top 7):
Rank 1: Song ID SOQARNP12A8C135A28 (Similarity: 0.9751)
Rank 2: Song ID SONWZEA12A6D4F9724 (Similarity: 0.9743)
Rank 3: Song ID SOCLHYJ12AB01820D4 (Similarity: 0.9738)
Rank 4: Song ID SOTSKKZ12A6D4F71C4 (Similarity: 0.9723)
Rank 5: Song ID SOZOVJG12AB018632B (Similarity: 0.9713)
Rank 6: Song ID SOZTGFT12A8C137271 (Similarity: 0.9687)
Rank 7: Song ID SOCYXEO12AB01821FA (Similarity: 0.9673)
